# Génération d’un patron de point de croix à partir d’une image

Ce notebook a pour objectif de transformer une image en **patron de point de croix**. Il va :
1. Permettre l’upload d’une image via un widget.
2. Envoyer cette image (ou un prompt associé) à l’API d’un service **Stable Diffusion** pour en obtenir une version “pixel art”.
3. Réduire la palette de couleurs et associer chaque couleur aux fils **DMC** les plus proches en s’appuyant sur un fichier JSON local.
4. Générer une grille adaptée, avec un symbole distinct pour chaque couleur, afin de faciliter la broderie.

Les étapes suivantes vous guideront pas à pas :
- **Installation des dépendances**  
- **Imports et configuration**  
- **Upload de l’image**  
- … *(à compléter ensuite)* …


## Installation / Mise à jour des dépendances

La cellule suivante permet d’installer ou de mettre à jour les librairies nécessaires :

- `numpy` pour le traitement en tableaux numériques.
- `pillow` pour la gestion des images (PIL).
- `requests` pour les requêtes HTTP (appel à l’API).
- `ipywidgets` pour les widgets interactifs.
- `jupyter-ui-poll` pour la boucle d’événements asynchrone dans Jupyter.


In [1]:
# Installe/Met à jour les bibliothèques manquantes.

%pip install numpy pillow requests ipywidgets jupyter-ui-poll --quiet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import requests
import base64
import json
import io
import os

from PIL import Image
from IPython.display import display
import ipywidgets as widgets

# Chargement du fichier local DMC_colors.json
# Chercher dans plusieurs emplacements possibles
dmc_paths = [
    "DMC_colors.json",
    os.path.join(os.path.dirname(os.path.abspath(".")), "..", "assets", "models", "DMC_colors.json"),
    "../../assets/models/DMC_colors.json",
]

DMC_COLORS = []
for dmc_path in dmc_paths:
    try:
        with open(dmc_path, "r", encoding="utf-8") as f:
            DMC_COLORS = json.load(f)
        print(f"Nombre de references DMC chargees : {len(DMC_COLORS)} (depuis {dmc_path})")
        break
    except FileNotFoundError:
        continue

if not DMC_COLORS:
    print("Fichier DMC_colors.json non trouve. Veuillez verifier son emplacement.")


## Paramètres de génération (img2img)

- **prompt** : C’est la description textuelle qui oriente la génération.  
- **steps** : Nombre d’itérations de sampling (plus haut = plus de détails, plus long).  
- **sampler_name** : Algorithme de sampling (ex. Euler a, DPM++ 2M, etc.).  
- **cfg_scale** : Facteur de guidance (plus c’est élevé, plus l’image colle au prompt, mais peut être moins créative).  
- **width / height** : Dimensions de l’image en pixels.  
- **denoising_strength** : Dans le contexte img2img, indique dans quelle mesure l’image initiale est altérée par la diffusion (0.0 = pas de changement, 1.0 = radicalement transformée).  
- **n_iter** / **batch_size** : Contrôlent le nombre d’images générées. Ici, nous renvoyons 3 images pour laisser le choix.  


In [3]:
from pathlib import Path
from dotenv import load_dotenv
import os

# Paramètre de base pour l’endpoint Stable Diffusion

# Chargement des variables d'environnement depuis .env
# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os
# Recherche du .env dans tous les parents (pour Papermill qui change le cwd)
current_path = Path.cwd()
env_loaded = False
while current_path.name != "GenAI" and len(current_path.parts) > 1:
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path}")
        env_loaded = True
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# Récupération de l'URL de l'API Stable Diffusion depuis .env
SD_BASE_URL = os.getenv("SD_BASE_URL", "https://stable-diffusion-webui-forge.yourdomain.com")

# Construct the full API URL by appending the endpoint path
SD_API_URL = f"{SD_BASE_URL}/sdapi/v1/img2img"

# Paramètres par défaut pour la requête (vous pouvez les affiner à la demande)
default_img2img_payload = {
    "prompt": "<lora:pixelbuildings128-v2:1> Pixel Art",
    "steps": 20,
    "sampler_name": "DPM++ 2M SDE",
    "scheduler": "karras",
    "cfg_scale": 7.5,
    "width": 1024,
    "height": 1024,
    "denoising_strength": 0.37,
    "override_settings": {
        "sd_model_checkpoint": "sd_xl_base_1.0"
    },
    # "seed": 2096377536
}


### Upload de l’image

Dans la prochaine cellule, vous pourrez sélectionner l’image que vous souhaitez transformer en patron de point de croix.

- Choisissez un fichier **.png**, **.jpg**, ou **.jpeg** depuis votre ordinateur.
- Le fichier sera ensuite stocké en mémoire dans le notebook (nous le lirons pour l’envoyer à l’API, ou pour l’afficher).

**Étapes suivantes** (à venir) :
- Nous enverrons l’image à notre endpoint Stable Diffusion pour obtenir la version "pixel art".
- Nous appliquerons ensuite une réduction de palette et un matching avec les codes DMC.
- Enfin, nous générerons la grille de points de croix.


In [ ]:
import time
import os

# Paramètres par défaut si non définis par Papermill
if 'notebook_mode' not in globals():
    # Détecter le mode batch via variable d'environnement
    notebook_mode = os.getenv("BATCH_MODE", "interactive").lower()
    if notebook_mode == "true":
        notebook_mode = "batch"

# ========================
# Variables de contrôle
# ========================
image_uploaded = False
generation_done = False
encoded_init_image = None
candidate_images = []  # contiendra la liste des images générées (PIL)
generated_image = None

# ========================
# Mode batch : créer une image de test et passer directement
# ========================
if notebook_mode in ("batch", "test"):
    print(f"Mode {notebook_mode.upper()} active : Creation d'une image de test.")
    
    # Creation d'une image de test (damier simple)
    test_img = Image.new('RGB', (1024, 1024), color='white')
    pixels = test_img.load()
    for i in range(1024):
        for j in range(1024):
            if (i // 64 + j // 64) % 2 == 0:
                pixels[i, j] = (0, 0, 0)  # Noir
            else:
                pixels[i, j] = (255, 0, 0)  # Rouge
                
    generated_image = test_img
    generation_done = True
    print(f"Image de test generee (1024x1024).")

else:
    # ========================
    # Mode interactif avec widgets
    # ========================
    from jupyter_ui_poll import ui_events
    import ipywidgets as widgets
    from IPython.display import display
    
    # Widgets pour l'upload et la generation
    upload_btn = widgets.FileUpload(
        accept='image/*',
        multiple=False,
        description='Uploader une image'
    )

    generate_btn = widgets.Button(
        description="Generer (img2img, 3 variantes)",
        button_style='success',
        icon='magic'
    )

    upload_output = widgets.Output()
    generate_output = widgets.Output()

    # Widgets pour la selection de l'image preferee
    selection_dropdown = widgets.Dropdown(
        options=[],
        description='Choix :',
        disabled=False
    )
    confirm_selection_btn = widgets.Button(
        description="Valider ce choix",
        button_style='info',
        icon='check'
    )
    selection_output = widgets.Output()

    # Callbacks
    def on_upload_change(change):
        global image_uploaded, encoded_init_image
        if upload_btn.value:
            file_info = upload_btn.value[0]
            file_bytes = file_info['content']
            encoded_init_image = base64.b64encode(file_bytes).decode('utf-8')
            with upload_output:
                upload_output.clear_output()
                print("Image chargee. Taille (octets) :", len(file_bytes))
                img = Image.open(io.BytesIO(file_bytes))
                display(img)
            image_uploaded = True

    def on_generate_click(button):
        global generation_done, candidate_images
        generation_done = False
        candidate_images = []

        if not image_uploaded or not encoded_init_image:
            with generate_output:
                generate_output.clear_output()
                print("Veuillez d'abord uploader une image.")
            return

        payload = default_img2img_payload.copy()
        payload["init_images"] = [encoded_init_image]
        payload["n_iter"] = 1
        payload["batch_size"] = 3

        with generate_output:
            generate_output.clear_output()
            print("Generation en cours (3 images)...")

            try:
                response = requests.post(url=SD_API_URL, json=payload, timeout=120)
                r = response.json()
                result_images = r.get('images', [])
                if not result_images:
                    print("Aucune image recue de l'API.")
                else:
                    print(f"{len(result_images)} image(s) generee(s).")
                    for idx, img_b64 in enumerate(result_images):
                        img_data = base64.b64decode(img_b64)
                        img_pil = Image.open(io.BytesIO(img_data))
                        candidate_images.append(img_pil)
                        display(img_pil)
                        print(f"Image #{idx}\n")
                    selection_dropdown.options = [
                        (f"Image #{i}", i) for i in range(len(candidate_images))
                    ]
                    selection_dropdown.value = 0
                    print("Choisissez votre image preferee ci-dessous.")
                generation_done = True
            except Exception as e:
                print(f"Erreur lors de l'appel API img2img : {e}")

    def on_confirm_selection_click(b):
        global generated_image
        with selection_output:
            selection_output.clear_output()
            if not candidate_images:
                print("Aucune image a selectionner.")
                return
            chosen_idx = selection_dropdown.value
            generated_image = candidate_images[chosen_idx]
            print(f"Vous avez choisi l'image #{chosen_idx}.")

    # Liaisons des callbacks
    upload_btn.observe(on_upload_change, names='value')
    generate_btn.on_click(on_generate_click)
    confirm_selection_btn.on_click(on_confirm_selection_click)

    # Affichage des widgets
    display(widgets.HTML("<h3>Etape : Chargement de l'image et generation (3 variantes)</h3>"))
    display(upload_btn)
    display(upload_output)
    display(generate_btn)
    display(generate_output)

    display(widgets.HTML("<h4>Etape : Selection de l'image preferee</h4>"))
    display(selection_dropdown)
    display(confirm_selection_btn)
    display(selection_output)

    print("En attente de l'upload puis du clic sur 'Generer' ...")

    with ui_events() as poll:
        while not generation_done:
            poll(1)
            time.sleep(0.1)

    print("Generation terminee !")


### Réduction de l’image en blocs de 8×8

Le modèle de diffusion nous donne une image de 1024×800 pixels, mais il s’agit visuellement d’un « pixel art » où chaque “carré” mesure 8×8 pixels.  
Pour simplifier le traitement, nous allons réduire l’image de la façon suivante :

1. **Parcourir** l’image d’entrée par blocs de 8×8.  
2. **Calculer la couleur moyenne** (ou dominante) de chaque bloc.  
3. **Construire** une nouvelle image dont la taille sera (largeur/8) × (hauteur/8).  
4. Chaque pixel de cette nouvelle image représentera le bloc 8×8 original.

Ce résultat final de 128×100 (si l’image initiale fait 1024×800) sera bien plus simple à manipuler pour la suite du matching couleurs (DMC) et la création de la grille finale.


In [ ]:
import numpy as np

BLOCK_SIZE = 8

def reduce_by_blocks(img: Image.Image, block_size: int = 8) -> Image.Image:
    """
    Réduit l'image en regroupant chaque bloc block_size x block_size
    en un seul pixel, basé sur la couleur moyenne de ce bloc.
    """
    # Convertir l'image en tableau numpy (RGBA ou RGB)
    arr = np.array(img.convert("RGB"))
    h, w, _ = arr.shape
    
    # Dimensions du résultat final
    new_w = w // block_size
    new_h = h // block_size
    
    # Tableau numpy pour accueillir le résultat
    small_arr = np.zeros((new_h, new_w, 3), dtype=np.uint8)
    
    # Parcours par blocs
    for y in range(new_h):
        for x in range(new_w):
            # Coordonnées du bloc
            y0 = y * block_size
            x0 = x * block_size
            
            # Sous-tableau correspondant au bloc (block_size x block_size x 3)
            block = arr[y0:y0+block_size, x0:x0+block_size, :]
            
            # Moyenne sur l'axe (0,1) => (hauteur, largeur du bloc)
            mean_color = block.mean(axis=(0,1))
            
            # On assigne au pixel (y, x)
            small_arr[y, x] = mean_color.astype(np.uint8)
    
    # Création de l'image PIL à partir du tableau small_arr
    small_img = Image.fromarray(small_arr, mode="RGB")
    return small_img

# Exemple d'utilisation
# On suppose que vous avez déjà une variable `generated_image` (la sortie stable diffusion).
# Si vous avez besoin de la relire depuis le disque, faites : generated_image = Image.open("nom_fichier.png")

if 'generated_image' in globals():
    reduced_image = reduce_by_blocks(generated_image, block_size=BLOCK_SIZE)
    
    # Affichage du résultat
    display(reduced_image)
    print(f"Nouvelle dimension : {reduced_image.width} x {reduced_image.height}")
else:
    print("⚠️ La variable 'generated_image' n'est pas définie. Veuillez définir ou charger l'image d'abord.")


### Matching des couleurs et conversion vers la palette DMC

Maintenant que nous avons une **image réduite** (par blocs 8×8) avec une dimension plus raisonnable (p. ex. 128×100), nous allons :

1. **Analyser chaque pixel** pour connaître sa couleur.  
2. Pour chacun de ces pixels (R, G, B), **trouver la couleur de fil DMC la plus proche**. Nous utiliserons un fichier `DMC_colors.json` contenant l’équivalence entre un code de fil (ex. 310, 498...) et un triplet (R, G, B).  
3. Cette étape nous donnera une image où chaque pixel est remplacé par la couleur “officielle” du fil DMC correspondant, et/ou un tableau d’index de couleurs.  
4. Enfin, nous pourrons générer la **grille de point de croix**, où chaque couleur DMC aura un **symbole** distinct pour la lisibilité.


In [ ]:
import math
import numpy as np
from PIL import Image

def build_dmc_lookup(dmc_list):
    """
    Pre-construit un tableau numpy des couleurs DMC et la liste des codes
    pour un matching vectorise rapide.
    """
    colors = []
    codes = []
    names = []
    for item in dmc_list:
        try:
            r = int(item["r"])
            g = int(item["g"])
            b = int(item["b"])
            colors.append([r, g, b])
            codes.append(str(item["floss"]))
            names.append(item.get("name", "Unknown"))
        except (ValueError, TypeError, KeyError):
            continue
    return np.array(colors, dtype=np.int32), codes, names


def convert_image_to_dmc(img: Image.Image, dmc_list):
    """
    Parcourt chaque pixel de l'image (img) et renvoie:
      - new_img: PIL Image recolorisee avec la couleur DMC la plus proche
      - dmc_codes: tableau 2D (h,w) des codes 'floss' (str)
    
    Utilise un matching vectorise numpy pour la performance.
    """
    arr_rgb = np.array(img.convert("RGB"), dtype=np.int32)
    h, w, _ = arr_rgb.shape

    # Pre-construire le lookup DMC
    dmc_colors, dmc_code_list, dmc_name_list = build_dmc_lookup(dmc_list)
    n_dmc = len(dmc_colors)

    # Reshape pour matching vectorise: (h*w, 3) vs (n_dmc, 3)
    pixels_flat = arr_rgb.reshape(-1, 3)  # (N, 3)
    
    # Calcul vectorise des distances euclidiennes au carre
    # pixels_flat[:, None, :] shape (N, 1, 3) - dmc_colors[None, :, :] shape (1, M, 3)
    # => diff shape (N, M, 3), squared sum => (N, M)
    diff = pixels_flat[:, None, :] - dmc_colors[None, :, :]
    distances = np.sum(diff * diff, axis=2)  # (N, M)
    
    # Index de la couleur DMC la plus proche pour chaque pixel
    best_indices = np.argmin(distances, axis=1)  # (N,)
    
    # Construire l'image DMC et les codes
    dmc_arr = dmc_colors[best_indices].reshape(h, w, 3).astype(np.uint8)
    dmc_codes = np.array([dmc_code_list[i] for i in best_indices], dtype=object).reshape(h, w)

    new_img = Image.fromarray(dmc_arr, mode="RGB")
    return new_img, dmc_codes


# Exemple d'utilisation
if 'reduced_image' in globals() and DMC_COLORS:
    print(f"Matching des couleurs DMC pour une image {reduced_image.width}x{reduced_image.height}...")
    dmc_image, dmc_codes_array = convert_image_to_dmc(reduced_image, DMC_COLORS)
    display(dmc_image)
    print("Exemple: code DMC du pixel (0,0) =", dmc_codes_array[0,0])
elif not DMC_COLORS:
    print("DMC_COLORS non charge - matching impossible.")
else:
    print("'reduced_image' n'est pas defini.")
